In [ ]:
import pickle
from bikeroot import Evaluation, hmls_solve
import os
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
import os, re, pickle, pandas as pd

def parse_results_file(path):
    """
    Read the result pickle at *path* and return a DataFrame with one row per
    (algorithm, trial) inside the file.

    Expected filename pattern:
        {loc}_{src}_{dst}_{criteria}.pkl
        e.g.  san_francisco_6928027683_9771126816_travel_distance_travel_time.pkl
    """
    fname = os.path.basename(path)

    # Match everything up to last 3 sections (src, dst, criteria) as location
    m = re.match(
        r"^(?P<loc>.+)_"              # location (greedy to include underscores)
        r"(?P<src>\d+)_"              # source node id
        r"(?P<dst>\d+)_"              # destination node id
        r"(?P<criteria>.+?)\.pkl$",   # criteria string
        fname,
    )
    if not m:
        raise ValueError(f"Filename {fname} does not match expected pattern.")
    meta = m.groupdict()
    meta["src"] = int(meta["src"])
    meta["dst"] = int(meta["dst"])

    with open(path, "rb") as f:
        data = pickle.load(f)

    rows = []
    for alg_name, alg_blob in data.items():
        for t_idx, trial in enumerate(alg_blob["trials"]):
            row = {
                **meta,
                "criteria_list": meta["criteria"].split("_"),
                "algorithm": alg_name,
                "trial_idx": t_idx,
                "n_paths": len(trial["paths"]),
                "time_sec": trial["time"],
                "hypervolume": float(trial["hypervolume"]),
            }
            for p_idx, cost_dict in enumerate(trial["costs"]):
                for crit, val in cost_dict.items():
                    row[f"cost_{crit}_{p_idx}"] = val
            rows.append(row)

    return pd.DataFrame(rows)


In [ ]:
all_dfs = []

for fname in os.listdir("data/testing/"):
    if fname.endswith(".pkl"):
        fpath = os.path.join("data/testing/", fname)
        try:
            df = parse_results_file(fpath)   # ← call the helper
            all_dfs.append(df)
            print(f"✓ parsed {fname}  →  {df.shape[0]} rows")
        except Exception as e:
            print(f"⚠️  Error processing {fname}: {e}")

# combine every file’s rows into one big DataFrame
results = (pd.concat(all_dfs, ignore_index=True)
             if all_dfs else pd.DataFrame())

print("\nFinal shape:", results.shape)

In [ ]:
# assuming `results` is the DataFrame produced earlier
avg_time_per_alg = (
    results
      .groupby("algorithm", as_index=False)["time_sec"]
      .mean()
      .sort_values("time_sec")        # optional: fastest → slowest
)

print(avg_time_per_alg)


In [ ]:
# assuming `results` is the DataFrame produced earlier
avg_hv_per_alg = (
    results
      .groupby("algorithm", as_index=False)["hypervolume"]
      .mean()
      .sort_values("hypervolume")        # optional: fastest → slowest
)

print(avg_hv_per_alg)


In [ ]:
import re
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd   # only to satisfy type hints; df already exists

def plot_distance_vs_safety_all(df: pd.DataFrame, *,
                                loc: str,
                                src: int,
                                dst: int,
                                criteria: str):
    """
    Scatter-plot *every* (travel-distance, safety) pair that was generated by
    each algorithm for one test-case file.

    Parameters
    ----------
    df       : the big tidy DataFrame (`results`)
    loc      : location string in the filename   (e.g. "spokane")
    src, dst : source and destination node IDs  (ints)
    criteria : criteria string from the filename
               ("travel_distance_travel_time_safety_comfort")
    """

    # ── 1. isolate the rows for this test case ────────────────────────────────
    mask = (
        (df["loc"] == loc) &
        (df["src"] == src) &
        (df["dst"] == dst) &
        (df["criteria"] == criteria)
    )
    sub = df.loc[mask]

    if sub.empty:
        raise ValueError("No matching rows – check loc/src/dst/criteria.")

    # ── 2. discover how many path-indices exist in these rows ─────────────────
    # columns look like "cost_travel distance_0", "cost_safety_0", ...
    dist_re   = re.compile(r"^cost_travel distance_(\d+)$")
    path_ids  = sorted({int(m.group(1))
                        for col in sub.columns
                        if (m := dist_re.match(col))})
    if not path_ids:
        raise KeyError("No travel-distance cost columns found.")

    # ── 3. plot every point (one colour per algorithm) ────────────────────────
    plt.figure(figsize=(6, 6))

    for alg, grp in sub.groupby("algorithm"):
        # accumulate all distance/safety pairs across *all* paths & trials
        d_vals, s_vals = [], []

        for p in path_ids:
            d_col = f"cost_travel distance_{p}"
            s_col = f"cost_safety_{p}"
            if d_col in grp.columns and s_col in grp.columns:
                # drop NaNs that arise when some trials have fewer paths
                d_vals.extend(grp[d_col].dropna().tolist())
                s_vals.extend(grp[s_col].dropna().tolist())

        if d_vals:         # skip if algorithm produced no valid pairs
            plt.scatter(d_vals, s_vals, label=alg)

    plt.xlabel("Travel distance")
    plt.ylabel("Safety")
    plt.title(f"{loc}  {src}→{dst}  ({criteria})\nAll path costs per algorithm")
    plt.grid(True)
    plt.legend(title="Algorithm")
    plt.tight_layout()
    plt.show()


# ── Example call ───────────────────────────────────────────────────────────────
plot_distance_vs_safety_all(
    results,
    loc      = "spokane",
    src      = 11053828893,
    dst      = 10757561389,
    criteria = "travel_distance_travel_time_safety_comfort"# change if you want the 1st, 2nd, … path in each trial
)


In [ ]:
plot_distance_vs_safety_all(
    #prague_581300652_3136859179
    results,
    loc      = "prague",
    src      = 581300652,
    dst      = 3136859179,
    criteria = "travel_distance_travel_time_safety_comfort"# change if you want the 1st, 2nd, … path in each trial
)

In [ ]:
# aveiro,3511264843,1278583286
plot_distance_vs_safety_all(
    #prague_581300652_3136859179
    results,
    loc      = "aveiro",
    src      = 3511264843,
    dst      = 1278583286,
    criteria = "travel_distance_travel_time_safety_comfort"# change if you want the 1st, 2nd, … path in each trial
)

In [ ]:
# aveiro,3511264843,1278583286
plot_distance_vs_safety_all(
    #prague_581300652_3136859179
    results,
    loc      = "aveiro",
    src      = 4961602591,
    dst      = 7706878440,
    criteria = "travel_distance_travel_time_safety_comfort"# change if you want the 1st, 2nd, … path in each trial
)

In [ ]:
plot_distance_vs_safety_all(
    results,
    loc      = "spokane",
    src      = 50939898,
    dst      = 12522426163,
    criteria = "travel_distance_travel_time_safety_comfort"# change if you want the 1st, 2nd, … path in each trial
)

In [ ]:
# 1️⃣  pick out just the HMLS rows
hmls_df = results[results["algorithm"] == "hmls"]

# 2️⃣  collapse trials so each (loc, src, dst) combo has ONE “solution count”
#     • here we keep the MAX n_paths seen across trials of that test
top_hmls = (
    hmls_df
      .groupby(["loc", "src", "dst"], as_index=False)["n_paths"]
      .max()
      .sort_values("n_paths", ascending=False)
)

# 3️⃣  print – adjust .head(10) for however many lines you want
print(top_hmls.head(15).to_string(index=False))
